**##** **IMPLEMENTING SELF ATTENTION WITH TRAINABLE WEIGHTS**

In [1]:
# Import the PyTorch library for tensor operations
import torch

# Create a tensor containing embedding vectors for six input words
inputs = torch.tensor(
  [[0.43, 0.15, 0.89],  # "Your" word embedding (x¹)
   [0.55, 0.87, 0.66],  # "journey" word embedding (x²)
   [0.57, 0.85, 0.64],  # "starts" word embedding (x³)
   [0.22, 0.58, 0.33],  # "with" word embedding (x⁴)
   [0.77, 0.25, 0.10],  # "one" word embedding (x⁵)
   [0.05, 0.80, 0.55]]  # "step" word embedding (x⁶)
)

Note that in GPT-like models, the input and output dimensions are usually the same.

But for illustration purposes, to better follow the computation, we choose different input **(d_in=3)**
and output **(d_out=2)** dimensions here.



In [2]:
# Select the second input token embedding ("journey")
x_2 = inputs[1]  # A

# Get the input embedding dimension (number of features)
d_in = inputs.shape[1]  # B

# Set the output dimension for Query, Key, and Value vectors
d_out = 2  # C

In [3]:
# Set a random seed to generate reproducible weight values
torch.manual_seed(123)

# Initialize the Query weight matrix (WQ)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

# Initialize the Key weight matrix (WK)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

# Initialize the Value weight matrix (WV)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [4]:
print(W_query)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])


In [5]:
print(W_key)

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])


In [6]:
print(W_value)

Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


In [7]:
# Compute the Query vector for the second input token
query_2 = x_2 @ W_query

# Compute the Key vector for the second input token
key_2 = x_2 @ W_key

# Compute the Value vector for the second input token
value_2 = x_2 @ W_value

# Display the generated Query vector
print(query_2)

tensor([0.4306, 1.4551])


In [8]:
# Generate Key vectors for all input tokens
keys = inputs @ W_key

# Generate Value vectors for all input tokens
values = inputs @ W_value

# Generate Query vectors for all input tokens
queries = inputs @ W_query

# Display the shape of the Key matrix
print("keys.shape:", keys.shape)

# Display the shape of the Value matrix
print("values.shape:", values.shape)

# Display the shape of the Query matrix
print("queries.shape:", queries.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])
queries.shape: torch.Size([6, 2])


In [9]:
# Select the Key vector corresponding to the second input token
keys_2 = keys[1]  # A

# Compute the attention score between the second Query and second Key
attn_score_22 = query_2.dot(keys_2)

# Display the calculated attention score
print(attn_score_22)

tensor(1.8524)


In [10]:
# Compute attention scores between the second Query and all Key vectors
attn_scores_2 = query_2 @ keys.T  # All attention scores for the given query

# Display the attention scores
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [11]:
# Compute pairwise attention scores between all Query and Key vectors
attn_scores = queries @ keys.T  # Attention score matrix (ω)

# Display the attention score matrix
print(attn_scores)

tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])


In [12]:
# Get the dimension of the Key vectors
d_k = keys.shape[-1]

# Apply scaled softmax to convert attention scores into attention weights
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)

# Display the attention weights for the second query
print(attn_weights_2)

# Display the Key vector dimension used for scaling
print(d_k)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
2


In [14]:
# Compute the context vector as the weighted sum of all Value vectors
context_vec_2 = attn_weights_2 @ values

# Display the context vector for the second query token
print(context_vec_2)

tensor([0.3061, 0.8210])


**In Structured Manner With Python Class**

In [15]:
# Import the PyTorch neural network module
import torch.nn as nn

# Define a Self-Attention layer from scratch
class SelfAttention_v1(nn.Module):

    # Initialize the self-attention layer and its learnable parameters
    def __init__(self, d_in, d_out):
        super().__init__()

        # Create the Query weight matrix
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))

        # Create the Key weight matrix
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))

        # Create the Value weight matrix
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    # Define the forward pass of the self-attention mechanism
    def forward(self, x):

        # Generate Key vectors from the input embeddings
        keys = x @ self.W_key

        # Generate Query vectors from the input embeddings
        queries = x @ self.W_query

        # Generate Value vectors from the input embeddings
        values = x @ self.W_value

        # Compute attention scores between all Query and Key pairs
        attn_scores = queries @ keys.T  # ω

        # Convert attention scores into attention weights using scaled softmax
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        # Compute context vectors as weighted sums of Value vectors
        context_vec = attn_weights @ values

        # Return the final context vectors
        return context_vec

In [16]:
# Set a random seed to ensure reproducible weight initialization
torch.manual_seed(123)

# Create an instance of the SelfAttention_v1 layer
sa_v1 = SelfAttention_v1(d_in, d_out)

# Pass the input embeddings through the self-attention layer
# to compute context vectors for all input tokens
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [17]:
# Define a self-attention layer using PyTorch Linear layers
class SelfAttention_v2(nn.Module):

    # Initialize the layer and optionally include bias terms
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()

        # Use nn.Linear to automatically perform matrix multiplication,
        # manage learnable weights (and optional biases), and simplify the code
        # Linear layer to generate Query vectors
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Linear layer to generate Key vectors
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Linear layer to generate Value vectors
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    # Define the forward pass of the self-attention mechanism
    def forward(self, x):

        # Generate Key vectors from the input embeddings
        keys = self.W_key(x)

        # Generate Query vectors from the input embeddings
        queries = self.W_query(x)

        # Generate Value vectors from the input embeddings
        values = self.W_value(x)

        # Compute attention scores between all Query and Key pairs
        attn_scores = queries @ keys.T

        # Convert attention scores into attention weights using scaled softmax
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        # Compute context vectors as weighted sums of Value vectors
        context_vec = attn_weights @ values

        # Return the final context vectors
        return context_vec

In [18]:
# Set a random seed to ensure reproducible weight initialization
torch.manual_seed(789)

# Create an instance of the SelfAttention_v2 layer
sa_v2 = SelfAttention_v2(d_in, d_out)

# Pass the input embeddings through the self-attention layer to generate context vectors for all input tokens
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
